# PhysAR-IDS -- NB16 (v6): Adversarial evaluation on the MSU / Morris dataset

**Purpose.** Paper 1 reports the attacker ordering (white-box > budgeted PPO ~ random > grey-box PGD) on one dataset only. This notebook runs the same four attackers on the MSU/Morris power-system data (NB9 split B, held-out passes) to see whether the ordering is a property of the method or of the Sweeten capture.

**Threat model.** Target: tuned XGBoost (NB9 parameters if `msu_xgb_tuned_params_v6.json` is present), 3-class (normal / natural / attack), trained on passes 1-10. The attacker perturbs the 4 relays' PMU measurements only (phasor magnitudes and angles, frequency, impedance); status flags, Z-inf flags and the 12 binary log columns are fixed. Budget: L-infinity, eps training-set standard deviations per feature, eps in {0.1, 0.3, 1.0}. Evasion = share of attack rows labelled normal or natural, **including rows the clean model already misses** (reported separately as `clean`).

**Deliberate choices.**
* The target uses PMU raw + flags + logs (no derived features), so perturbing a raw measurement needs no recomputation of derived columns. Check the NB9 ablation (`all` vs `no_derived`) and state this in the paper.
* Logs are binary and almost always 0, so a sigma budget is meaningless for them; they are held fixed.
* No physical-consistency constraint is enforced on the perturbed PMU values (same limitation as Paper 1).
* Grey-box PGD: MLP surrogate trained on the target's hard labels of the training rows. Random search: 10 random +/- eps vertices per row (same 10 queries as PPO). Budgeted PPO: RecurrentPPO, LSTM 64, 10 steps, hard-label oracle, 50k timesteps. White-box: greedy search over every attacker-controlled feature using the tree thresholds (a lower bound on evasion).
* Same evaluation rows for every attacker and seed: up to 20 rows from each test attack run. Intervals are 95% bootstrap over **runs** (rows inside a run are correlated), averaged over seeds.
* Seeds [42, 7, 123, 2024, 99]; paired t-tests over seeds with Holm correction within each budget.

Run after NB9, with `msu_combined.csv`. Needs `sb3-contrib` and `gymnasium`.

In [ ]:
import os, time, json, warnings
import numpy as np, pandas as pd, joblib
import matplotlib; matplotlib.use('Agg') if os.environ.get('NB16_HEADLESS') else None
import matplotlib.pyplot as plt
from scipy import stats
import xgboost as xgb, torch, torch.nn as nn
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')
PIPELINE_VERSION = 'v6'
SEEDS = [int(s) for s in os.environ.get('NB16_SEEDS', '42,7,123,2024,99').split(',')]
EPS_LIST = [0.1, 0.3, 1.0]          # per-feature L-inf budget in training-set standard deviations
PPO_EPS = [0.1]                     # budgets at which a PPO agent is trained (add 0.3 for a second budget)
PPO_TIMESTEPS = int(os.environ.get('NB16_PPO_STEPS', 50000)); PPO_MAX_STEPS = 10
N_QUERIES = PPO_MAX_STEPS           # random search gets the same number of oracle queries per row as PPO
PGD_ITERS = 10; K_CAND = None; WB_ITERS = 25   # None: white-box considers every attacker-controlled feature (a 12-feature greedy search was far too weak: 0.68 vs 0.95 at 0.1 sigma in a check)
ROWS_PER_RUN = 20                   # evaluation rows per test attack run (same rows for every attacker)
N_BOOT = 2000
CSV_PATH = r'C:\Users\user\Desktop\smartrid1\msu_combined.csv'
if not os.path.exists(CSV_PATH):
    for alt in ['msu_combined.csv', '/mnt/user-data/uploads/msu_combined.csv', r'C:\Users\user\Desktop\smartrid1\SmartGrid_Dissertation\msu_combined.csv']:
        if os.path.exists(alt): CSV_PATH = alt; break
assert os.path.exists(CSV_PATH), 'set CSV_PATH'
OUT_DIR = os.environ.get('NB16_OUT', os.path.dirname(os.path.abspath(CSV_PATH))); os.makedirs(OUT_DIR, exist_ok=True); os.chdir(OUT_DIR)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
XGB_DEV = dict(tree_method='hist', device='cuda') if torch.cuda.is_available() and not os.environ.get('NB16_CPU') else dict(tree_method='hist')
print('xgboost', xgb.__version__, '| torch', torch.__version__, DEV, '| seeds', SEEDS, '| out', os.getcwd())


In [ ]:
raw = pd.read_csv(CSV_PATH); marker = raw['marker'].astype(int).values
RELAYS = ['R1', 'R2', 'R3', 'R4']
LOG_COLS = [c for c in raw.columns if c != 'marker' and not c.startswith('R')]
PMU_COLS = [c for c in raw.columns if c.startswith('R')]
X = raw.drop(columns='marker').astype('float64').copy(); FLAG_COLS = []
for r in RELAYS:                                   # identical cleaning to NB9
    z = f'{r}-PA:Z'; bad = ~np.isfinite(X[z].values)
    X[z] = np.where(bad, X.loc[~bad, z].max(), X[z]); X[f'{r}_Z_inf'] = bad.astype(float); FLAG_COLS.append(f'{r}_Z_inf')
FEATS = PMU_COLS + FLAG_COLS + LOG_COLS            # target model uses NO derived features (see markdown)
Xv = X[FEATS].values.astype('float32'); F = len(FEATS)
NATURAL = {1, 2, 3, 4, 5, 6, 13, 14}; NORMAL = {41}
g3 = np.where(np.isin(marker, list(NORMAL)), 0, np.where(np.isin(marker, list(NATURAL)), 1, 2)); G3 = ['normal', 'natural', 'attack']
start = np.r_[True, marker[1:] != marker[:-1]]; run_id = np.cumsum(start) - 1; run_marker = marker[start]
pass_of_run = np.zeros(run_id.max() + 1, int); seen = {}
for r_, m_ in enumerate(run_marker): pass_of_run[r_] = seen.get(m_, 0); seen[m_] = pass_of_run[r_] + 1
pass_id = pass_of_run[run_id]
tr = np.flatnonzero(pass_id < 10); te = np.flatnonzero(pass_id >= 10)        # NB9 split B: held-out passes
mu = Xv[tr].mean(0); sd = Xv[tr].std(0)
status_idx = {FEATS.index(f'{r}:S') for r in RELAYS}
PERT = np.array([i for i, f in enumerate(FEATS) if f in PMU_COLS and sd[i] > 1e-9 and i not in status_idx])   # attacker-controlled PMU measurements
MASK = np.zeros(F, bool); MASK[PERT] = True; sd1 = np.where(sd > 1e-9, sd, 1.0).astype('float32')
LO, HI = Xv[tr].min(0), Xv[tr].max(0)
print(f'{len(tr)} train / {len(te)} test rows | {F} target features | {len(PERT)} attacker-controlled PMU features | logs, flags, status fixed')
# evaluation rows: up to ROWS_PER_RUN attack rows from every test run (same rows for all attackers and seeds)
rng0 = np.random.RandomState(0); ev = []
for r_ in np.unique(run_id[te]):
    idx = te[(run_id[te] == r_) & (g3[te] == 2)]
    if len(idx): ev.append(rng0.choice(idx, min(ROWS_PER_RUN, len(idx)), replace=False))
EV = np.sort(np.concatenate(ev)); print('evaluation rows', len(EV), '| attack runs', len(np.unique(run_id[EV])), '| markers', len(np.unique(marker[EV])))


In [ ]:
try:
    BEST = json.load(open(f'msu_xgb_tuned_params_{PIPELINE_VERSION}.json'))['params']; print('tuned params from NB9')
except Exception:
    BEST = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.6, min_child_weight=5, reg_lambda=1.0); print('NB9 params not found, using fallback')
def fit_target(seed):
    p = dict(BEST); p.update(random_state=seed, n_jobs=-1, verbosity=0, **XGB_DEV)
    return xgb.XGBClassifier(**p).fit(Xv[tr], g3[tr])
def proba(m, A): return m.get_booster().inplace_predict(np.ascontiguousarray(A, dtype=np.float32), predict_type='value') if False else m.predict_proba(np.ascontiguousarray(A, dtype=np.float32))
def evaded(m, A): return proba(m, A).argmax(1) != 2


In [ ]:
def to_raw(z): return z * sd1 + mu
def to_z(A): return (A - mu) / sd1

class Surrogate(nn.Module):
    def __init__(s): super().__init__(); s.n = nn.Sequential(nn.Linear(F, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 3))
    def forward(s, x): return s.n(x)
def train_surrogate(target, seed, epochs=12):
    torch.manual_seed(seed); net = Surrogate().to(DEV); opt = torch.optim.Adam(net.parameters(), 1e-3)
    Z = torch.tensor(np.clip(to_z(Xv[tr]), -10, 10), dtype=torch.float32, device=DEV)
    y = torch.tensor(target.predict(Xv[tr]), dtype=torch.long, device=DEV)         # grey-box: attacker only sees target's hard labels
    for ep in range(epochs):
        perm = torch.randperm(len(Z), device=DEV)
        for i in range(0, len(Z), 512):
            b = perm[i:i + 512]; opt.zero_grad(); nn.functional.cross_entropy(net(Z[b]), y[b]).backward(); opt.step()
    agree = (net(Z).argmax(1) == y).float().mean().item(); return net.eval(), agree

def atk_pgd(net, rows, eps):
    z0 = torch.tensor(np.clip(to_z(Xv[rows]), -10, 10), dtype=torch.float32, device=DEV); m = torch.tensor(MASK, device=DEV).float()
    z = z0.clone(); alpha = eps / 4
    for _ in range(PGD_ITERS):
        z.requires_grad_(True); lg = net(z)
        loss = torch.logsumexp(lg[:, :2], 1) - lg[:, 2]                              # push the attack logit below the benign logits
        g, = torch.autograd.grad(loss.sum(), z)
        z = (z.detach() + alpha * g.sign() * m); z = z0 + (z - z0).clamp(-eps, eps)
    delta = (z.detach() - z0).cpu().numpy().astype('float32')
    return (Xv[rows] + delta * sd1 * MASK).astype('float32')

def atk_random(target, rows, eps, seed):
    rng = np.random.RandomState(seed); base = Xv[rows]; ok = evaded(target, base); out = base.copy(); q = np.zeros(len(rows))
    for k in range(N_QUERIES):
        todo = np.flatnonzero(~ok)
        if not len(todo): break
        sgn = rng.choice([-1.0, 1.0], size=(len(todo), len(PERT))).astype('float32')   # random vertex of the L-inf ball
        cand = base[todo].copy(); cand[:, PERT] += eps * sd1[PERT] * sgn
        e = evaded(target, cand); out[todo[e]] = cand[e]; ok[todo[e]] = True; q[todo] += 1
    return out

def tree_thresholds(target):
    df = target.get_booster().trees_to_dataframe(); df = df[df.Feature != 'Leaf']
    fid = {f'f{i}': i for i in range(F)} if str(df.Feature.iloc[0]).startswith('f') and not str(df.Feature.iloc[0]) in FEATS else {f: i for i, f in enumerate(FEATS)}
    th = {}
    for f, g in df.groupby('Feature'): th[fid[f]] = np.unique(g.Split.values.astype('float32'))
    return th
def gain_rank(target):
    sc = target.get_booster().get_score(importance_type='total_gain'); out = []
    for k, v in sc.items():
        i = int(k[1:]) if k.startswith('f') and k[1:].isdigit() else FEATS.index(k)
        if MASK[i]: out.append((v, i))
    return [i for v, i in sorted(out, reverse=True)]
def atk_whitebox(target, rows, eps, k_cand=K_CAND, iters=WB_ITERS, clip_range=False):
    th = tree_thresholds(target); cand_feats = gain_rank(target)[:k_cand]
    x0 = Xv[rows].copy(); lo = x0 - eps * sd1; hi = x0 + eps * sd1
    if clip_range: lo = np.maximum(lo, LO); hi = np.minimum(hi, HI)
    x = x0.copy(); pa = proba(target, x)[:, 2]; done = pa < 0
    done = proba(target, x).argmax(1) != 2
    for it in range(iters):
        act = np.flatnonzero(~done)
        if not len(act): break
        C = []                                                   # candidates per active row: lo, hi, next threshold up, next threshold down per candidate feature
        for f in cand_feats:
            t = th.get(f, np.array([], 'float32')); v = x[act, f]
            j = np.searchsorted(t, v, side='right'); up = np.where(j < len(t), t[np.minimum(j, len(t) - 1)] if len(t) else v, v)
            j2 = np.searchsorted(t, v, side='left') - 1; dn = np.where(j2 >= 0, np.nextafter(t[np.maximum(j2, 0)], np.float32(-np.inf)) if len(t) else v, v)
            for val in (lo[act, f], hi[act, f], np.clip(up, lo[act, f], hi[act, f]), np.clip(dn, lo[act, f], hi[act, f])): C.append((f, val.astype('float32')))
        n = len(C); big = np.repeat(x[act][:, None, :], n, axis=1)
        for c, (f, val) in enumerate(C): big[:, c, f] = val
        P = proba(target, big.reshape(-1, F))[:, 2].reshape(len(act), n)
        best = P.argmin(1); bp = P[np.arange(len(act)), best]; improve = bp < pa[act] - 1e-9
        for r_i in np.flatnonzero(improve):
            f, val = C[best[r_i]]; x[act[r_i], f] = val[r_i]
        pa[act[improve]] = bp[improve]; done[act[~improve]] = True                  # stuck rows stop
        done[act] |= (proba(target, x[act]).argmax(1) != 2)
    return x

# ----- PPO (budgeted, hard-label oracle) -----
import gymnasium as gym
from gymnasium import spaces
class EvadeEnv(gym.Env):
    def __init__(self, target, pool, eps, seed):
        super().__init__(); self.t = target; self.pool = pool; self.eps = eps; self.rng = np.random.RandomState(seed)
        self.booster = target.get_booster(); P = len(PERT)
        self.action_space = spaces.Box(-1, 1, (P,), np.float32); self.observation_space = spaces.Box(-10, 10, (F + 3 + 1,), np.float32)
    def _q(self, x): return int(self.booster.inplace_predict(x[None, :].astype(np.float32)).argmax())
    def _obs(self): return np.concatenate([np.clip(to_z(self.x), -10, 10), np.eye(3)[self.lab], [self.k / PPO_MAX_STEPS]]).astype(np.float32)
    def reset(self, seed=None, options=None):
        self.x0 = Xv[self.rng.choice(self.pool)].copy(); self.x = self.x0.copy(); self.k = 0; self.lab = self._q(self.x); return self._obs(), {}
    def step(self, a):
        d = np.clip((self.x - self.x0)[PERT] / sd1[PERT] + a * self.eps, -self.eps, self.eps); self.x[PERT] = self.x0[PERT] + d * sd1[PERT]
        self.k += 1; self.lab = self._q(self.x); ev_ = self.lab != 2
        r = (10.0 if ev_ else -1.0) - 0.01 * float((a ** 2).sum()); trunc = self.k >= PPO_MAX_STEPS
        return self._obs(), r, bool(ev_), bool(trunc and not ev_), {}
def atk_ppo(target, rows, eps, seed):
    from sb3_contrib import RecurrentPPO
    pool = tr[g3[tr] == 2]; pool = np.random.RandomState(seed).choice(pool, min(20000, len(pool)), replace=False)
    env = EvadeEnv(target, pool, eps, seed)
    model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0,
                         policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu').learn(PPO_TIMESTEPS)
    x0 = Xv[rows]; x = x0.copy(); ok = evaded(target, x0); state = None; start_ = np.ones(len(rows), bool); booster = target.get_booster()
    for k in range(PPO_MAX_STEPS):
        lab = proba(target, x).argmax(1)
        obs = np.concatenate([np.clip(to_z(x), -10, 10), np.eye(3)[lab], np.full((len(rows), 1), k / PPO_MAX_STEPS)], 1).astype(np.float32)
        a, state = model.predict(obs, state=state, episode_start=start_, deterministic=True); start_ = np.zeros(len(rows), bool)
        d = np.clip((x - x0)[:, PERT] / sd1[PERT] + a * eps, -eps, eps); xn = x.copy(); xn[:, PERT] = x0[:, PERT] + d * sd1[PERT]
        new = (proba(target, xn).argmax(1) != 2)
        x = np.where(ok[:, None], x, xn); ok |= new                                    # rows that already evaded are frozen
    return x


In [ ]:
def disp_ok(x, rows, eps):                                                              # sanity: perturbation stays inside the budget and only touches PERT
    d = np.abs(x - Xv[rows]) / sd1; assert d[:, ~MASK].max() == 0, 'non-controllable feature changed'; return float(d.max())
RES = {}; ROWSREC = {}; INFO = {}; t0 = time.time()
for s in SEEDS:
    target = fit_target(s); pred_te = target.predict(Xv[te]); f1c = f1_score(g3[te], pred_te, average='macro')
    clean_miss = float((target.predict(Xv[EV]) != 2).mean()); net, agree = train_surrogate(target, s)
    INFO[s] = dict(clean_macro_f1=f1c, clean_miss_eval=clean_miss, surrogate_agreement=agree); print(f'seed {s}: clean macro-F1 {f1c:.4f}, clean miss on eval {clean_miss:.4f}, surrogate agreement {agree:.3f} ({time.time()-t0:.0f}s)')
    for eps in EPS_LIST:
        for name, fn in (('PGD_grey', lambda: atk_pgd(net, EV, eps)), ('Random', lambda: atk_random(target, EV, eps, s)),
                         ('WhiteBox', lambda: atk_whitebox(target, EV, eps)), ('WhiteBox_rangeclip', lambda: atk_whitebox(target, EV, eps, clip_range=True))):
            xa = fn(); assert disp_ok(xa, EV, eps) <= eps + 1e-3, name
            e = evaded(target, xa); RES[(s, name, eps)] = float(e.mean()); ROWSREC[(s, name, eps)] = e
            print(f'   {name:19s} eps={eps}: evasion {e.mean():.4f}')
    for eps in PPO_EPS:
        xa = atk_ppo(target, EV, eps, s); assert disp_ok(xa, EV, eps) <= eps + 1e-3
        e = evaded(target, xa); RES[(s, 'PPO_budgeted', eps)] = float(e.mean()); ROWSREC[(s, 'PPO_budgeted', eps)] = e; print(f'   PPO_budgeted        eps={eps}: evasion {e.mean():.4f} ({time.time()-t0:.0f}s)')
    ROWSREC[(s, 'clean', 0.0)] = (target.predict(Xv[EV]) != 2)


In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
R = pd.DataFrame([dict(seed=s, attacker=a, eps=e, evasion=v) for (s, a, e), v in RES.items()])
tab = R.groupby(['attacker', 'eps']).evasion.agg(['mean', 'std']).round(4); print(tab.to_string())
runs = run_id[EV]; uruns = np.unique(runs); grp = {r_: np.flatnonzero(runs == r_) for r_ in uruns}
def burst_ci(key_list, seed_boot=0):
    M = np.mean([ROWSREC[k].astype(float) for k in key_list], 0); rb = np.random.RandomState(seed_boot); out = []
    for _ in range(N_BOOT):
        pick = rb.choice(uruns, len(uruns), replace=True); out.append(np.concatenate([M[grp[r_]] for r_ in pick]).mean())
    return float(M.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))
CI = {}
for (a, e) in sorted({(a, e) for (_, a, e) in RES}, key=lambda t: (t[1], t[0])):
    CI[(a, e)] = burst_ci([(s, a, e) for s in SEEDS if (s, a, e) in ROWSREC])
CI[('clean', 0.0)] = burst_ci([(s, 'clean', 0.0) for s in SEEDS])
cit = pd.DataFrame([dict(attacker=a, eps=e, evasion=v[0], ci_lo=v[1], ci_hi=v[2]) for (a, e), v in CI.items()]); print('\nburst-clustered 95% CI (bootstrap over runs):'); print(cit.round(4).to_string(index=False))
pairs = []
for e in [x for x in EPS_LIST]:
    piv = R[R.eps == e].pivot(index='seed', columns='attacker', values='evasion')
    fam = [('WhiteBox', 'Random'), ('WhiteBox', 'PGD_grey'), ('Random', 'PGD_grey')] + ([('PPO_budgeted', 'Random'), ('PPO_budgeted', 'PGD_grey'), ('WhiteBox', 'PPO_budgeted')] if e in PPO_EPS else [])
    ps = []
    for a, b in fam:
        if len(SEEDS) > 1: t, p = stats.ttest_rel(piv[a], piv[b])
        else: p = np.nan
        ps.append(p); pairs.append(dict(eps=e, a=a, b=b, diff=(piv[a] - piv[b]).mean(), p_raw=p))
    k = len(ps); pairs[-k:] = [dict(d, p_holm=h) for d, h in zip(pairs[-k:], holm(np.nan_to_num(ps, nan=1.0)))]
PT = pd.DataFrame(pairs); print('\npaired t-tests over seeds (Holm within budget):'); print(PT.round(4).to_string(index=False))


In [ ]:
PM = pd.DataFrame({a: pd.Series(np.mean([ROWSREC[(s, a, 0.1)] for s in SEEDS], 0)).groupby(marker[EV]).mean() for a in ['PGD_grey', 'Random', 'WhiteBox'] + (['PPO_budgeted'] if 0.1 in PPO_EPS else [])})
print('\nper attack-scenario evasion at 0.1 sigma (mean over seeds); 10 most resistant to white-box:'); print(PM.sort_values('WhiteBox').head(10).round(3).to_string())
# white-box search-strength check: more candidate features on a 400-row subset, seed 42 model (is the 12-feature greedy search a weak lower bound?)
tgt = fit_target(SEEDS[0]); sub = np.random.RandomState(1).choice(EV, min(400, len(EV)), replace=False); SENS = {}
for kc in (12, 24, len(PERT)):   # shows how much a restricted greedy search under-reports
    SENS[kc] = float(evaded(tgt, atk_whitebox(tgt, sub, 0.1, k_cand=kc)).mean()); print(f'white-box 0.1 sigma, K_CAND={kc}: {SENS[kc]:.4f}')


In [ ]:
T = {'attacks': R, 'summary': tab.reset_index(), 'burst_ci': cit, 'paired': PT, 'per_marker': PM.reset_index()}
for k_, t_ in T.items(): t_.to_csv(f'msu_adv_{k_}_{PIPELINE_VERSION}.csv', index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'EPS_LIST': EPS_LIST, 'PPO_EPS': PPO_EPS, 'tables': T, 'burst_ci': CI, 'seed_info': INFO,
             'n_pert': len(PERT), 'n_eval_rows': len(EV), 'whitebox_kcand_check': SENS, 'rows_evaded': ROWSREC, 'eval_rows': EV, 'run_id_eval': runs},
            f'msu_adv_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB16', 'seeds': SEEDS, 'xgboost': xgb.__version__, 'ppo_timesteps': PPO_TIMESTEPS, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb16_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=1)
print('saved', [f for f in os.listdir('.') if f.startswith('msu_adv') or f.startswith('nb16')])
